# Experiment 1 — Debug Notebook

Replicates `experiment1_denoiser_ablation.py` with the full `dag_pnp` algorithm **inlined** for step-by-step debugging.

Setup: SF4 graph, d=20, n=500, 1 seed.  
Methods: NOTEARS · DAGMA · Greedy-BIC · DAG-PnP (with GDSSDiffusionDenoiser)

## 0. Imports & configuration

In [1]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import defaultdict
from typing import Callable, Dict, Optional, Tuple
from scipy.linalg import expm

# ── make sure we import from dag-pnp/, not a parent directory ──
DAG_PNP_DIR = os.path.abspath('.')
if DAG_PNP_DIR not in sys.path:
    sys.path.insert(0, DAG_PNP_DIR)

print('Working dir:', DAG_PNP_DIR)

Working dir: /home/nicolas/GDSS/dag-pnp


In [29]:
# ── Hyperparameters (mirrors dag_config.py) ──────────────────────────────────

WEIGHT_LOW  = 0.2
WEIGHT_HIGH = 0.5
K_EDGES     = 4      # SF4: 4 parents per node ≈ 4d edges
THRESHOLD   = 0.2    # edge-detection threshold (fixed)

# experiment setup
D         = 20
N_SAMPLES = 20
SEED      = 10        # rng = seed * 1000 = 0
NORMALISE = False

CKPT_PATH = 'gdss/checkpoints/sem_SF_d20/Apr16-01:12:06.pth'

# ── adaptive s & W_max (from dag_config.shared_kwargs) ───────────────────────

def dagma_s():
    rho_bound = 2.0 * K_EDGES * WEIGHT_HIGH ** 2
    return float(1.5 * rho_bound + 1.0)

def dagma_W_max(d):
    expected_edges = K_EDGES * d
    mean_w2 = (WEIGHT_LOW**2 + WEIGHT_LOW * WEIGHT_HIGH + WEIGHT_HIGH**2) / 3.0
    return float(3.0 * np.sqrt(expected_edges * mean_w2) + 1.0)

def shared_kwargs(d):
    return dict(
        num_iters   = 1000,
        lambda_data = 1.0,
        lambda_dag  = 0.02,
        eta         = 0.005,
        c           = 0.3,
        epsilon     = 1e-6,
        s           = dagma_s(),
        alpha       = 1.0,
        sigma_min   = 0.0,
        sigma_max   = 1.0,
        smooth_beta = 0.0,
        W_max       = dagma_W_max(d),
        delta_min   = 1e-6,
    )

kw = shared_kwargs(D)
S  = kw['s']
print(f'd={D}  s={S:.3f}  W_max={kw["W_max"]:.3f}  lambda_dag={kw["lambda_dag"]}')

d=20  s=4.000  W_max=10.675  lambda_dag=0.02


## 1. Data generation

In [30]:
# ── copy of data_utils.py ────────────────────────────────────────────────────

def scale_free_dag(d, m=4, weight_low=0.5, weight_high=1.0, seed=None):
    rng   = np.random.default_rng(seed)
    order = rng.permutation(d)
    W     = np.zeros((d, d))
    indegree = np.zeros(d)
    for rank_j in range(1, d):
        n_parents = min(m, rank_j)
        weights   = indegree[:rank_j] + 1.0
        probs     = weights / weights.sum()
        parent_ranks = rng.choice(rank_j, size=n_parents, replace=False, p=probs)
        for pr in parent_ranks:
            i = order[pr]; j = order[rank_j]
            w = rng.uniform(weight_low, weight_high) * rng.choice([-1, 1])
            W[i, j] = w
            indegree[pr] += 1
    return W


def sample_sem(W, n, noise_std=1.0, seed=None):
    rng = np.random.default_rng(seed)
    d   = W.shape[0]
    E   = rng.standard_normal((n, d)) * noise_std
    return E @ np.linalg.inv(np.eye(d) - W)


def to_tensor(arr, device='cpu', dtype=torch.float32):
    return torch.tensor(arr, device=device, dtype=dtype)


def normalise_X(X):
    std_X  = X.std(dim=0).clamp(min=1e-8)
    X_opt  = X / std_X
    denorm = (std_X.unsqueeze(0) / std_X.unsqueeze(1)).numpy()
    return X_opt, denorm


def denormalise_W(W_norm, denorm):
    return W_norm * denorm


def threshold_W(W, threshold=0.3):
    return (np.abs(W) > threshold).astype(int)


def evaluate_dag(W_pred, W_true, threshold=0.3):
    B_pred = threshold_W(W_pred, threshold)
    B_true = threshold_W(W_true, threshold)
    tp = int(np.sum((B_pred == 1) & (B_true == 1)))
    fp = int(np.sum((B_pred == 1) & (B_true == 0)))
    fn = int(np.sum((B_pred == 0) & (B_true == 1)))
    shd = fp + fn
    tpr       = tp / (tp + fn + 1e-9)
    precision = tp / (tp + fp + 1e-9)
    fdr       = fp / (tp + fp + 1e-9)
    f1        = 2 * precision * tpr / (precision + tpr + 1e-9)
    return dict(shd=shd, tpr=float(tpr), fdr=float(fdr),
                precision=float(precision), f1=float(f1),
                tp=tp, fp=fp, fn=fn)


print('Data utilities defined.')

Data utilities defined.


In [31]:
# ── Generate one SF4 graph ────────────────────────────────────────────────────
rng_seed = SEED * 1000   # = 0

W_true = scale_free_dag(D, m=K_EDGES,
                        weight_low=WEIGHT_LOW, weight_high=WEIGHT_HIGH,
                        seed=rng_seed)
X_np   = sample_sem(W_true, N_SAMPLES, seed=rng_seed + 1)
X      = to_tensor(X_np)

if NORMALISE:
    X_opt, denorm = normalise_X(X)
else:
    X_opt  = X
    denorm = np.ones((D, D), dtype=np.float32)

n_edges_true = int(np.sum(np.abs(W_true) > THRESHOLD))
print(f'W_true: {D}x{D}, {n_edges_true} edges (threshold={THRESHOLD})')
print(f'X_np:   shape={X_np.shape},  std range [{X_np.std(0).min():.2f}, {X_np.std(0).max():.2f}]')

W_true: 20x20, 70 edges (threshold=0.2)
X_np:   shape=(20, 20),  std range [0.79, 1.78]


## 2. Loss functions (inlined from losses_dag.py)

In [25]:
# ── losses_dag.py inlined ─────────────────────────────────────────────────────

def data_loss(X, W):
    """L_data = 1/(2n) ||X - XW||_F^2"""
    n = X.shape[0]
    return 0.5 / n * torch.sum((X - X @ W) ** 2)

def grad_data_loss(X, W):
    n = X.shape[0]
    return -1.0 / n * (X.T @ (X - X @ W))

def _M(W, s):
    d = W.shape[0]
    return s * torch.eye(d, device=W.device, dtype=W.dtype) - W * W

def dagma_penalty(W, s=1.0):
    """h(W) = -log det(sI - W∘W) + d log s   (= 0 iff W is a DAG)"""
    d = W.shape[0]
    sign, logdet = torch.linalg.slogdet(_M(W, s))
    if sign.item() <= 0:
        return torch.tensor(float('inf'), device=W.device, dtype=W.dtype)
    return -logdet + d * np.log(s)

def grad_dagma(W, s=1.0):
    return 2.0 * torch.linalg.inv(_M(W, s)).T * W

def grad_lext(X, W, lambda_data, lambda_dag, s=1.0):
    return lambda_data * grad_data_loss(X, W) + lambda_dag * grad_dagma(W, s)

print('Loss functions defined.')

Loss functions defined.


## 3. DAG-PnP algorithm (inlined from dag_pnp.py)

In [26]:
# ── Step-size rule (eq. 11) ───────────────────────────────────────────────────

def barrier_step_size(W, grad, s, eta, c, epsilon):
    """
    gamma = min( eta,  c * delta(W) / (||∇L_ext||_F + epsilon) )
    delta(W) = sigma_min(sI - W∘W)
    """
    d     = W.shape[0]
    M     = s * torch.eye(d, device=W.device, dtype=W.dtype) - W * W
    delta = torch.linalg.svdvals(M).min().item()
    grad_norm = torch.norm(grad, 'fro').item()
    barrier   = c * delta / (grad_norm + epsilon)
    return float(min(eta, barrier))


# ── Noise schedule (eq. 12) ───────────────────────────────────────────────────

def noise_schedule(X, W, alpha, sigma_min, sigma_max, sigma_prev, smooth_beta):
    """
    sigma^2 = alpha * ||X - XW||^2_F / ||X||^2_F,  clipped to [sigma_min, sigma_max]
    """
    residual_sq = torch.sum((X - X @ W) ** 2).item()
    data_sq     = torch.sum(X ** 2).item() + 1e-12
    sigma = float(np.sqrt(alpha * residual_sq / data_sq))
    sigma = float(np.clip(sigma, sigma_min, sigma_max))
    if sigma_prev is not None and smooth_beta > 0.0:
        sigma = smooth_beta * sigma_prev + (1.0 - smooth_beta) * sigma
    return sigma


# ── Projection onto K (eq. 9) ─────────────────────────────────────────────────

def project_onto_K(W, s, delta_min, W_max):
    """
    K = { W : det(sI - W∘W) >= delta_min, ||W||_F <= W_max, diag(W) = 0 }
    """
    W = W.clone()
    d = W.shape[0]
    I = torch.eye(d, device=W.device, dtype=W.dtype)
    W.fill_diagonal_(0.0)
    norm = torch.norm(W, 'fro').item()
    if norm > W_max:
        W = W * (W_max / norm)
    for _ in range(300):
        M = s * I - W * W
        sign, logdet = torch.linalg.slogdet(M)
        det = (sign * torch.exp(logdet)).item()
        if det >= delta_min:
            break
        W = W * 0.9
    return W


print('DAG-PnP helper functions defined.')

DAG-PnP helper functions defined.


## 4. Load GDSSDiffusionDenoiser

In [7]:
from dag_pnp import GDSSDiffusionDenoiser, IdentityDenoiser

print(f'Loading checkpoint: {CKPT_PATH}')
denoiser = GDSSDiffusionDenoiser.from_checkpoint(CKPT_PATH, device='cpu')
print('Denoiser loaded.')

# quick sanity: denoiser should return a (D,D) float tensor
W_test = torch.randn(D, D) * 0.3
out    = denoiser(W_test, sigma=0.3)
print(f'  denoiser output: shape={out.shape}  dtype={out.dtype}  '
      f'unique={out.unique().numel()}  (should be d*d={D*D})')

Loading checkpoint: gdss/checkpoints/sem_SF_d20/Apr16-01:12:06.pth
Denoiser loaded.
  denoiser output: shape=torch.Size([20, 20])  dtype=torch.float32  unique=400  (should be d*d=400)


## 5. Single-iteration debug walkthrough

Run one iteration manually and inspect every intermediate tensor.

In [8]:
# ── manual single iteration ───────────────────────────────────────────────────
s         = kw['s']
eta       = kw['eta']
c         = kw['c']
epsilon   = kw['epsilon']
delta_min = kw['delta_min']
W_max     = kw['W_max']
lambda_data = kw['lambda_data']
lambda_dag  = kw['lambda_dag']
alpha       = kw['alpha']
sigma_min   = kw['sigma_min']
sigma_max   = kw['sigma_max']
smooth_beta = kw['smooth_beta']

# initialise W
W_dbg = torch.zeros(D, D)
W_dbg = project_onto_K(W_dbg, s, delta_min, W_max)

# ─── Step 1: gradient ─────────────────────────────────────────────────────────
grad = grad_lext(X_opt, W_dbg, lambda_data, lambda_dag, s)
print('Step 1 — gradient')
print(f'  ||grad||_F = {torch.norm(grad, "fro").item():.4f}')
print(f'  grad min/max: {grad.min().item():.4f} / {grad.max().item():.4f}')

# ─── Step 2: step size ────────────────────────────────────────────────────────
M     = s * torch.eye(D) - W_dbg * W_dbg
delta = torch.linalg.svdvals(M).min().item()
gamma = barrier_step_size(W_dbg, grad, s, eta, c, epsilon)
print(f'\nStep 2 — step size')
print(f'  delta(W) = {delta:.4f}')
print(f'  gamma    = {gamma:.4e}  (eta={eta}, barrier={c*delta/(torch.norm(grad,"fro").item()+epsilon):.4e})')

# ─── Step 3: gradient step ────────────────────────────────────────────────────
z = W_dbg - gamma * grad
z.fill_diagonal_(0.0)
print(f'\nStep 3 — gradient step z = W - gamma*grad')
print(f'  ||z||_F   = {torch.norm(z, "fro").item():.4f}')
print(f'  z min/max : {z.min().item():.4f} / {z.max().item():.4f}')

# ─── Step 4: noise schedule ───────────────────────────────────────────────────
sigma_k = noise_schedule(X_opt, W_dbg, alpha, sigma_min, sigma_max, None, smooth_beta)
print(f'\nStep 4 — noise schedule')
print(f'  sigma_k = {sigma_k:.4f}  (range [{sigma_min}, {sigma_max}])')

# ─── Step 5: denoising ────────────────────────────────────────────────────────
W_struct = denoiser(z, sigma_k)
print(f'\nStep 5 — denoiser output')
print(f'  ||W_struct||_F   = {torch.norm(W_struct, "fro").item():.4f}')
print(f'  W_struct min/max : {W_struct.min().item():.4f} / {W_struct.max().item():.4f}')
n_edges_struct = int((W_struct.abs() > THRESHOLD).sum().item())
print(f'  edges @ threshold={THRESHOLD}: {n_edges_struct}  (true: {n_edges_true})')

# ─── Step 6: projection ───────────────────────────────────────────────────────
W_proj = project_onto_K(W_struct, s, delta_min, W_max)
sign, logdet = torch.linalg.slogdet(s * torch.eye(D) - W_proj * W_proj)
det_proj = (sign * torch.exp(logdet)).item()
print(f'\nStep 6 — projection')
print(f'  ||W_proj||_F = {torch.norm(W_proj, "fro").item():.4f}  (W_max={W_max:.2f})')
print(f'  det(sI-W∘W)  = {det_proj:.4e}  (must be >= delta_min={delta_min})')

Step 1 — gradient
  ||grad||_F = 10.1247
  grad min/max: -2.0830 / 1.0260

Step 2 — step size
  delta(W) = 4.0000
  gamma    = 5.0000e-03  (eta=0.005, barrier=1.1852e-01)

Step 3 — gradient step z = W - gamma*grad
  ||z||_F   = 0.0371
  z min/max : -0.0051 / 0.0047

Step 4 — noise schedule
  sigma_k = 1.0000  (range [0.0, 1.0])

Step 5 — denoiser output
  ||W_struct||_F   = 5.7890
  W_struct min/max : -2.0942 / 1.7413
  edges @ threshold=0.2: 50  (true: 70)

Step 6 — projection
  ||W_proj||_F = 3.2168  (W_max=10.67)
  det(sI-W∘W)  = 1.0966e+12  (must be >= delta_min=1e-06)


## 6. Full run — all baselines

In [10]:
# ── NOTEARS baseline (inlined from baselines.py) ──────────────────────────────

def notears_baseline(X, lambda_data=1.0, lambda_dag=0.02, num_iters=1000,
                     lr=0.001, W_max=10.0, grad_clip=10.0, W_init=None):
    X_np = X.numpy() if isinstance(X, torch.Tensor) else np.array(X)
    n, d = X_np.shape
    W    = np.zeros((d, d)) if W_init is None else W_init.copy()
    np.fill_diagonal(W, 0.0)
    history = {'loss': [], 'h': []}
    for _ in range(num_iters):
        residual  = X_np - X_np @ W
        grad_data = -1.0 / n * (X_np.T @ residual)
        WW = np.clip(W * W, None, 50.0)
        E  = expm(WW)
        grad_h = 2.0 * E.T * W
        grad_h = np.where(np.isfinite(grad_h), grad_h, 0.0)
        grad   = lambda_data * grad_data + lambda_dag * grad_h
        g_norm = np.linalg.norm(grad, 'fro')
        if g_norm > grad_clip:
            grad = grad * (grad_clip / g_norm)
        W = W - lr * grad
        np.fill_diagonal(W, 0.0)
        w_norm = np.linalg.norm(W, 'fro')
        if w_norm > W_max:
            W = W * (W_max / w_norm)
        history['loss'].append(float(0.5 / n * np.sum(residual ** 2)))
        h_val = float(np.trace(E) - d)
        history['h'].append(h_val if np.isfinite(h_val) else float('inf'))
    return W, history


# ── Greedy BIC baseline (inlined from baselines.py) ──────────────────────────

def _bic_score(X_np, j, parents, penalty=1.0):
    n = X_np.shape[0]
    if len(parents) == 0:
        rss = float(np.sum(X_np[:, j] ** 2))
    else:
        Pa  = X_np[:, parents]
        beta, _, _, _ = np.linalg.lstsq(Pa, X_np[:, j], rcond=None)
        rss = float(np.sum((X_np[:, j] - Pa @ beta) ** 2))
    return -(n/2.0) * np.log(max(rss, 1e-12) / n) - penalty * len(parents) * np.log(n) / 2.0


def _is_acyclic(adj):
    d = adj.shape[0]
    in_deg = adj.sum(axis=0).copy()
    queue  = [i for i in range(d) if in_deg[i] == 0]
    n_vis  = 0
    while queue:
        u = queue.pop(); n_vis += 1
        for v in range(d):
            if adj[u, v]:
                in_deg[v] -= 1
                if in_deg[v] == 0: queue.append(v)
    return n_vis == d


def graph_gen_baseline(X, max_parents=5, penalty=1.0):
    X_np = X.numpy() if isinstance(X, torch.Tensor) else np.array(X)
    n, d = X_np.shape
    adj     = np.zeros((d, d), dtype=bool)
    parents = [[] for _ in range(d)]
    scores  = [_bic_score(X_np, j, [], penalty) for j in range(d)]
    history = {'bic_gains': []}
    while True:
        best_gain = 0.0; best_i = best_j = -1; best_new = 0.0
        for i in range(d):
            for j in range(d):
                if i == j or adj[i, j]: continue
                if max_parents and len(parents[j]) >= max_parents: continue
                adj[i, j] = True
                if not _is_acyclic(adj):
                    adj[i, j] = False; continue
                new_s = _bic_score(X_np, j, parents[j] + [i], penalty)
                gain  = new_s - scores[j]
                if gain > best_gain:
                    best_gain = gain; best_i = i; best_j = j; best_new = new_s
                adj[i, j] = False
        if best_i == -1: break
        adj[best_i, best_j] = True
        parents[best_j].append(best_i)
        scores[best_j] = best_new
        history['bic_gains'].append(float(best_gain))
    W_est = np.zeros((d, d))
    for j in range(d):
        if not parents[j]: continue
        Pa   = X_np[:, parents[j]]
        beta, _, _, _ = np.linalg.lstsq(Pa, X_np[:, j], rcond=None)
        for ki, i in enumerate(parents[j]):
            W_est[i, j] = beta[ki]
    return W_est, history


print('Baseline functions defined.')

Baseline functions defined.


In [11]:
# ── NOTEARS ───────────────────────────────────────────────────────────────────
print('Running NOTEARS...')
W_n_norm, hist_n = notears_baseline(
    X_opt, lambda_data=kw['lambda_data'], lambda_dag=kw['lambda_dag'],
    num_iters=kw['num_iters'], lr=0.001, W_max=kw['W_max'], grad_clip=10.0,
)
W_notears = denormalise_W(W_n_norm, denorm)
m_n = evaluate_dag(W_notears, W_true, THRESHOLD)
print(f'  SHD={m_n["shd"]}  F1={m_n["f1"]:.3f}  TP={m_n["tp"]}  FP={m_n["fp"]}  FN={m_n["fn"]}')

Running NOTEARS...
  SHD=61  F1=0.430  TP=23  FP=14  FN=47


In [ ]:
# ── Greedy-BIC ────────────────────────────────────────────────────────────────
print('Running Greedy-BIC...')
W_g, hist_g = graph_gen_baseline(X_opt, max_parents=min(5, D // 2))
W_bic = denormalise_W(W_g, denorm)
m_g = evaluate_dag(W_bic, W_true, THRESHOLD)
print(f'  SHD={m_g["shd"]}  F1={m_g["f1"]:.3f}  TP={m_g["tp"]}  FP={m_g["fp"]}  FN={m_g["fn"]}')

In [33]:
# ── Main DAG-PnP loop (Algorithm 1) ─────────────────────────────────────────

def dag_pnp_debug(
    X, denoiser,
    num_iters=500, lambda_data=1.0, lambda_dag=1.0,
    eta=0.01, c=0.3, epsilon=1e-6, s=1.0,
    alpha=1.0, sigma_min=0.01, sigma_max=0.5, smooth_beta=0.9,
    W_max=10.0, delta_min=1e-6,
    W_init=None, project=True, inject_noise=True,
    verbose=True, log_every=20,
    debug_iters=None,
):
    """
    DAG-PnP Algorithm 1 with explicit VP-SDE renoising + SNR-based mixing.

    When inject_noise=True and denoiser has .sde (GDSSDiffusionDenoiser):
      - Loop iterates over VP-SDE alpha_t schedule: t goes from T -> t_min
      - After gradient step z, renoise:
            z_tilde = mean_coeff(t_k) * z  +  std(t_k) * eps
      - Tweedie denoising on pre-noised z_tilde:
            W_struct = (z_tilde + std^2 * score(z_tilde, t_k)) / mean_coeff(t_k)
      - Denoiser influence is weighted by the current SNR:
            denoiser_weight = sigma_k / sigma_start   (1 early, 0 late)
            W_tilde = (1 - denoiser_weight) * z  +  denoiser_weight * W_struct
        Early (sigma_k ~ sigma_start): W_tilde ~ W_struct  (explore DAG prior)
        Late  (sigma_k ~ 0):           W_tilde ~ z          (exploit data fidelity)

    This mirrors the reference annealing: the denoiser guides exploration when
    the signal is noisy and hands off to the gradient as the signal clears.
    """
    d      = X.shape[1]
    device = X.device
    dtype  = X.dtype

    W = torch.zeros(d, d, device=device, dtype=dtype) if W_init is None else W_init.clone()
    W = project_onto_K(W, s, delta_min, W_max)

    history = {'loss': [], 'h': [], 'sigma': [], 'gamma': [], 'denoiser_w': []}
    debug_snapshots = []
    debug_iter_set = set() if debug_iters is None else set(debug_iters)

    # ── VP-SDE time schedule: t_k from T down to t_min over num_iters steps ──
    _has_sde = inject_noise and hasattr(denoiser, 'sde')
    if _has_sde:
        t_vals = np.linspace(float(denoiser.sde.T), denoiser.t_min, num_iters)
        # sigma_start = std(T) — the max noise level, used for normalisation
        _dummy = torch.zeros(1, 1, 1, device=device)
        _, _std0 = denoiser.sde.marginal_prob(_dummy, torch.tensor([t_vals[0]], device=device))
        sigma_start = _std0.item()

    if verbose:
        print(f'DAG-PnP | d={d}  n={X.shape[0]}  iters={num_iters}')
        print(f'  lambda_data={lambda_data}  lambda_dag={lambda_dag}')
        print(f'  eta={eta}  c={c}  s={s:.3f}  W_max={W_max:.2f}')
        if _has_sde:
            _, _std_end = denoiser.sde.marginal_prob(_dummy, torch.tensor([t_vals[-1]], device=device))
            print(f'  noise: VP-SDE t[{t_vals[0]:.3f} -> {t_vals[-1]:.4f}]'
                  f'  =>  sigma[{sigma_start:.3f} -> {_std_end.item():.4f}]')
            print(f'  mixing: denoiser_weight = sigma_k / sigma_start  (1->0)')
        print('-' * 65)

    for k in range(num_iters):

        # ── Step 1: gradient of L_ext ──────────────────────────────────────────
        grad = grad_lext(X, W, lambda_data, lambda_dag, s)

        # ── Step 2: barrier-aware step size ───────────────────────────────────
        gamma = barrier_step_size(W, grad, s, eta, c, epsilon)

        # ── Step 3: gradient step ─────────────────────────────────────────────
        z = W - (gamma) * grad

        # ── Step 6: projection ────────────────────────────────────────────────
        # z = project_onto_K(z, s, delta_min, W_max) if project else W_tilde
        # ── Steps 4–5: renoising + denoising + SNR-weighted mixing ────────────
        if _has_sde:
            # Step 4 — VP-SDE renoising:
            #   z_tilde = mean_coeff(t_k) * z  +  std(t_k) * eps
            t_k    = torch.tensor([t_vals[k]], device=device, dtype=dtype)
            z_b    = z.unsqueeze(0)                                           # (1, d, d)
            mean_z, std_k = denoiser.sde.marginal_prob(z_b, t_k)             # (1,d,d), (1,)
            z_tilde = (mean_z + std_k.view(-1, 1, 1) * torch.randn_like(z_b)).squeeze(0)
            sigma_k = std_k.item()

            # Step 5 — Tweedie denoising on z_tilde (no extra noise):
            #   W_struct = (z_tilde + std^2 * score(z_tilde, t_k)) / mean_coeff(t_k)
            with torch.no_grad():
                t_d    = torch.tensor([t_vals[k]], device=denoiser.device, dtype=torch.float32)
                adj_t  = z_tilde.unsqueeze(0).to(denoiser.device, torch.float32)
                x_feat = torch.ones(1, d, denoiser.max_feat_num, device=denoiser.device)
                flags  = torch.ones(1, d, device=denoiser.device)
                mean_c, std_td = denoiser.sde.marginal_prob(torch.ones_like(adj_t), t_d)
                score   = denoiser.score_fn(x_feat, adj_t, flags, t_d)
                std_sq  = (std_td ** 2).view(1, 1, 1)
                W_struct = ((adj_t + std_sq * score) / mean_c).squeeze(0).cpu()

            # Step 5b — SNR-based mixing:
            #   denoiser_weight = sigma_k / sigma_start   ∈ (0, 1]
            #   High sigma (early): denoiser_weight ~ 1  → W_tilde ~ W_struct
            #   Low  sigma (late):  denoiser_weight ~ 0  → W_tilde ~ z
            # denoiser_w = sigma_k / sigma_start
            denoiser_w = 1 * (mean_c[0,0,0].item() / std_td.item())**-1  # more principled, but can be >1 early on
            W_tilde = (1.0 - denoiser_w) * z + denoiser_w * W_struct.to(device)
            # denoiser_w = 1.0
            # W_tilde = z

        elif inject_noise:
            sigma_k    = noise_schedule(X, W, alpha, sigma_min, sigma_max,
                                        history['sigma'][-1] if history['sigma'] else None,
                                        smooth_beta)
            denoiser_w = 1.0
            z_tilde    = z
            W_struct   = denoiser(z, sigma_k)
            W_tilde    = W_struct

        else:
            sigma_k    = 0.0
            denoiser_w = 0.0
            z_tilde    = z
            W_struct   = z
            W_tilde    = z

        # ── Step 6: projection ────────────────────────────────────────────────
        W = project_onto_K(W_tilde, s, delta_min, W_max) if project else W_tilde
        # W = W_tilde
        # ── Logging ───────────────────────────────────────────────────────────
        loss_val = data_loss(X, W).item()
        h_val    = dagma_penalty(W, s).item()
        history['loss'].append(loss_val)
        history['h'].append(h_val)
        history['sigma'].append(sigma_k)
        history['gamma'].append(gamma)
        history['denoiser_w'].append(denoiser_w)

        if k in debug_iter_set:
            debug_snapshots.append(dict(
                k=k, sigma=sigma_k, denoiser_w=denoiser_w, gamma=gamma,
                W=W.detach().clone(),
                z=z.detach().clone(),
                z_tilde=z_tilde.detach().clone() if hasattr(z_tilde, 'detach') else torch.tensor(z_tilde),
                W_struct=W_struct.detach().clone() if hasattr(W_struct, 'detach') else W_struct.clone(),
            ))

        if verbose and (k % log_every == 0 or k == num_iters - 1):
            print(f'  iter {k:4d}/{num_iters} | loss={loss_val:.5f}  h={h_val:.5f}'
                  f'  dw={denoiser_w:.3f}  sigma={sigma_k:.4f}  gamma={gamma:.3e}')

    if verbose:
        print('-' * 65)
        print('Done.')

    return W, history, debug_snapshots


print('dag_pnp_debug defined.')

# ── DAG-PnP (GDSSDiffusionDenoiser) ──────────────────────────────────────────
print('Running DAG-PnP...')
DEBUG_ITERS = [0, 1, 5, 10, 50, 100, 499, 999]

# ── DAGMA (identity denoiser = no structural prior) ───────────────────────────
print('Running DAGMA...')
kw['eta'] = 0.1
W_d_norm, hist_d, _ = dag_pnp_debug(
    X_opt, IdentityDenoiser(), inject_noise=False,
    verbose=True, log_every=100, **kw
)
W_dagma = denormalise_W(W_d_norm.detach().numpy(), denorm)
m_d = evaluate_dag(W_dagma, W_true, THRESHOLD)
print(f'  SHD={m_d["shd"]}  F1={m_d["f1"]:.3f}  TP={m_d["tp"]}  FP={m_d["fp"]}  FN={m_d["fn"]}')

W_p_norm, hist_p, snapshots = dag_pnp_debug(
    X_opt, denoiser, inject_noise=True,
    verbose=True, log_every=100,
    debug_iters=DEBUG_ITERS,
    **kw
)
W_pnp = denormalise_W(W_p_norm.detach().numpy(), denorm)
m_p = evaluate_dag(W_pnp, W_true, THRESHOLD)
print(f'  SHD={m_p["shd"]}  F1={m_p["f1"]:.3f}  TP={m_p["tp"]}  FP={m_p["fp"]}  FN={m_p["fn"]}')

dag_pnp_debug defined.
Running DAG-PnP...
Running DAGMA...
DAG-PnP | d=20  n=20  iters=1000
  lambda_data=1.0  lambda_dag=0.02
  eta=0.1  c=0.3  s=4.000  W_max=10.67
-----------------------------------------------------------------
  iter    0/1000 | loss=7.67656  h=0.00031  dw=0.000  sigma=0.0000  gamma=9.464e-02
  iter  100/1000 | loss=1.29671  h=0.16858  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  200/1000 | loss=0.91063  h=0.40475  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  300/1000 | loss=0.72838  h=0.67310  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  400/1000 | loss=0.61849  h=0.96297  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  500/1000 | loss=0.54518  h=1.26298  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  600/1000 | loss=0.49395  h=1.54766  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  700/1000 | loss=0.45745  h=1.78057  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  800/1000 | loss=0.43098  h=1.94444  dw=0.000  sigma=0.0000  gamma=1.000e-01
  iter  900/

## 7. Results summary table

In [ ]:
results = {
    'NOTEARS':     m_n,
    'DAGMA':       m_d,
    'Greedy-BIC':  m_g,
    'DAG-PnP':     m_p,
}

print(f'\n{"Method":<14}  {"SHD":>5}  {"F1":>6}  {"TPR":>6}  {"FDR":>6}  {"TP":>4}  {"FP":>4}  {"FN":>4}')
print('-' * 62)
for method, m in results.items():
    print(f'{method:<14}  {m["shd"]:>5}  {m["f1"]:>6.3f}  {m["tpr"]:>6.3f}  '
          f'{m["fdr"]:>6.3f}  {m["tp"]:>4}  {m["fp"]:>4}  {m["fn"]:>4}')
print(f'\n(GT edges: {n_edges_true}  threshold={THRESHOLD}  d={D}  n={N_SAMPLES}  seed={SEED})')

## 8. Visualisations

In [ ]:
# ── W_true vs W_estimated ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

w_mats = {
    'W_true':    W_true,
    'NOTEARS':   W_notears,
    'DAGMA':     W_dagma,
    'Greedy-BIC':W_bic,
    'DAG-PnP':   W_pnp,
}

vmax = max(np.abs(W_true).max(), 0.5)
for ax, (name, W_) in zip(axes, w_mats.items()):
    im = ax.imshow(W_, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    m_ = evaluate_dag(W_, W_true, THRESHOLD) if name != 'W_true' else None
    title = name
    if m_ is not None:
        title += f'\nSHD={m_["shd"]}  F1={m_["f1"]:.2f}'
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('child j'); ax.set_ylabel('parent i')
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(f'SF4  d={D}  n={N_SAMPLES}  seed={SEED}', fontsize=12)
plt.tight_layout()
plt.savefig('exp1_W_comparison.png', dpi=120)
plt.show()
print('Saved exp1_W_comparison.png')

In [ ]:
# ── Binary edge maps (thresholded) ────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, (name, W_) in zip(axes, w_mats.items()):
    B = threshold_W(W_, THRESHOLD)
    ax.imshow(B, cmap='Greys', vmin=0, vmax=1, aspect='auto')
    n_e = int(B.sum())
    ax.set_title(f'{name}\n{n_e} edges', fontsize=10)
    ax.set_xlabel('child j'); ax.set_ylabel('parent i')

fig.suptitle(f'Thresholded adjacency (thr={THRESHOLD})  — GT={n_edges_true} edges', fontsize=12)
plt.tight_layout()
plt.savefig('exp1_B_comparison.png', dpi=120)
plt.show()

In [ ]:
# ── Convergence curves ────────────────────────────────────────────────────────
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

# loss curves
ax1.plot(hist_n['loss'], label='NOTEARS', alpha=0.8)
ax1.plot(hist_d['loss'], label='DAGMA',   alpha=0.8)
ax1.plot(hist_p['loss'], label='DAG-PnP', alpha=0.8)
ax1.set_xlabel('iteration'); ax1.set_ylabel('L_data'); ax1.set_title('Data loss')
ax1.legend(); ax1.set_yscale('log')

# acyclicity penalty
h_d_clipped = [min(v, 50) for v in hist_d['h']]
h_p_clipped = [min(v, 50) for v in hist_p['h']]
ax2.plot(h_d_clipped, label='DAGMA',   alpha=0.8)
ax2.plot(h_p_clipped, label='DAG-PnP', alpha=0.8)
ax2.set_xlabel('iteration'); ax2.set_ylabel('h(W) (clipped at 50)')
ax2.set_title('DAGMA penalty h(W)')
ax2.legend()

# noise schedule (DAG-PnP)
ax3.plot(hist_p['sigma'], color='purple', alpha=0.8)
ax3.set_xlabel('iteration'); ax3.set_ylabel('sigma_k')
ax3.set_title('DAG-PnP noise schedule')

plt.tight_layout()
plt.savefig('exp1_convergence.png', dpi=120)
plt.show()

In [ ]:
# ── Denoiser snapshots at selected iterations ─────────────────────────────────
if snapshots:
    n_snap = len(snapshots)
    fig, axes = plt.subplots(3, n_snap, figsize=(3 * n_snap, 8))
    if n_snap == 1:
        axes = axes[:, np.newaxis]

    vmax_s = 0.6
    for col, snap in enumerate(snapshots):
        k     = snap['k']
        z_np  = snap['z'].numpy()
        ws_np = snap['W_struct'].numpy()
        w_np  = snap['W'].numpy()

        for row, (mat, title) in enumerate([
            (z_np,  f'z (grad step)'),
            (ws_np, f'W_struct (denoised)'),
            (w_np,  f'W (projected)'),
        ]):
            ax = axes[row, col]
            im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax_s, vmax=vmax_s, aspect='auto')
            if row == 0:
                ax.set_title(f'iter={k}\nσ={snap["sigma"]:.3f}', fontsize=9)
            if col == 0:
                ax.set_ylabel(title, fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle('Denoiser snapshots: z → W_struct → W_projected', fontsize=11)
    plt.tight_layout()
    plt.savefig('exp1_denoiser_snapshots.png', dpi=100)
    plt.show()
else:
    print('No debug snapshots captured (set debug_iters in cell above).')

## 9. Step-by-step denoiser diagnostics

Inspect what the denoiser does at each snapshot: how many edges survive after thresholding, weight distribution, diagonal check.

In [ ]:
print(f'{'iter':>5}  {'sigma':>6}  {'||z||':>7}  {'||W_s||':>8}  '
      f'{"edges_z":>8}  {"edges_Ws":>9}  {"diag_mean_Ws":>13}')
print('-' * 72)

for snap in snapshots:
    k      = snap['k']
    sigma  = snap['sigma']
    z_np   = snap['z'].numpy()
    ws_np  = snap['W_struct'].numpy()

    norm_z  = float(np.linalg.norm(z_np,  'fro'))
    norm_ws = float(np.linalg.norm(ws_np, 'fro'))
    edges_z  = int((np.abs(z_np)  > THRESHOLD).sum())
    edges_ws = int((np.abs(ws_np) > THRESHOLD).sum())
    diag_mean = float(np.abs(np.diag(ws_np)).mean())

    print(f'{k:>5}  {sigma:>6.4f}  {norm_z:>7.3f}  {norm_ws:>8.3f}  '
          f'{edges_z:>8}  {edges_ws:>9}  {diag_mean:>13.4f}')

print(f'\nGT edges: {n_edges_true}')